In [ ]:
# | default_exp content.checks

# Content Checks
> Stateless validators: given content → return SEO issues.

In [ ]:
# | export
from seootter.parser.extractors import extract_headers


In [ ]:
# | export
def check_h1_count(headers: list[dict],
                   title: str | None = None,
                   title_is_h1: bool = False
                   ) -> dict:
    "Check H1 count — for Quarto, the title frontmatter field acts as H1."
    if title_is_h1 and title: return {"h1_count": 1, "has_single_h1": True, "h1_source": "title"}
    h1s = [h for h in headers if h["type"] == "h1"]
    return {"h1_count": len(h1s), "has_single_h1": len(h1s) == 1}

In [ ]:
# | export
def keyword_in_first_section(content: str,
                             keyword: str,
                             percent: int = 10
                             ) -> bool:
    "Check if keyword appears in the first X% of content."
    return keyword.lower() in content[:int(len(content) * percent / 100)].lower()

In [ ]:
# | export
def check_paragraph_length(content: str
                           ) -> dict:
    "Check average number of sentences per paragraph."
    try:
        paragraphs = [p.strip() for p in content.split("\n\n") if p.strip()]
        avg = sum(len(p.split(". ")) for p in paragraphs) / len(paragraphs) if paragraphs else 0
        return {"avg_sentences_per_paragraph": avg, "is_optimal": 2 <= avg <= 4}
    except Exception:
        return {"avg_sentences_per_paragraph": 0, "is_optimal": False}

In [ ]:
# | export
def keyword_in_metadata(metadata: dict,
                        keyword: str,
                        desc_field: str = "description"
                        ) -> dict:
    "Check if keyword appears in title and description metadata fields."
    kw = keyword.lower()
    return {"in_title": kw in str(metadata.get("title", "")).lower(),
            "in_description": kw in str(metadata.get(desc_field, "")).lower()}


In [ ]:
# | export
def keyword_in_alt_texts(images: list[dict],
                         keyword: str
                         ) -> bool:
    "Check if keyword appears in any image alt text."
    return any(keyword.lower() in img.get("alt_text", "").lower() for img in images)

In [ ]:
# | export
def analyze_header_distribution(headers: list[dict]
                                ) -> dict:
    "Analyze header type distribution as counts and percentages."
    from collections import Counter
    counts = dict(Counter(h["type"] for h in headers))
    total = len(headers)
    return {"counts": counts,
            "percentages": {k: (v / total * 100) if total > 0 else 0 for k, v in counts.items()}}


In [ ]:
# | export
def check_keyword_placement(keyword: str | None,
                            metadata: dict,
                            headers: list[dict],
                            content: str,
                            url: str,
                            desc_field: str = "description",
                            title_is_h1: bool = False
                            ) -> dict:
    "Check where the focus keyword appears across page elements."
    try:
        if not keyword:
            return {"in_h1": False, "in_url": False,
                    "keyword_in_meta": {"in_title": False, "in_description": False},
                    "keyword_at_first": False}
        in_h1 = any(keyword.lower() in h["content"].lower() for h in headers if h["type"] == "h1")
        if title_is_h1: in_h1 = in_h1 or keyword.lower() in str(metadata.get("title", "")).lower()
        return {"in_h1": in_h1, "in_url": keyword.lower() in url.lower(),
                "keyword_in_meta": keyword_in_metadata(metadata, keyword, desc_field=desc_field),
                "keyword_at_first": keyword_in_first_section(content, keyword)}
    except Exception:
        return {"in_h1": False, "in_url": False,
                "keyword_in_meta": {"in_title": False, "in_description": False},
                "keyword_at_first": False}

In [ ]:
  # | export

def content_freshness(last_updated: str,
                      days: int = 180
                      ) -> dict:
    "Check content freshness based on last updated date."
    from datetime import datetime
    try:
        updated = datetime.fromisoformat(str(last_updated))
        delta = (datetime.now() - updated).days
        return {"last_updated": last_updated, "days_since_update": delta, "is_fresh": delta <= days}
    except (ValueError, TypeError):
        return {"last_updated": last_updated, "days_since_update": None, "is_fresh": True}
